In [1]:
import os
from ares import logger

os.chdir("../")

In [2]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataProcessingConfig:
    root_dir: Path
    data_dir: Path
    train: Path
    test: Path
    geocode_cache: Path

In [3]:
from ares.constants import *
from ares.utils.common import read_yaml, load_json, save_json, create_directories

In [4]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_processing_config(self) -> DataProcessingConfig:
        config = self.config.data_processing

        create_directories([config.root_dir])

        data_processing_config = DataProcessingConfig(
            root_dir=config.root_dir,
            data_dir=config.data_dir,
            train=config.train,
            test=config.test,
            geocode_cache=Path(config.geocode_cache),
        )
        return data_processing_config


In [5]:
import pandas as pd
import numpy as np
import googlemaps
from dotenv import load_dotenv

load_dotenv()
maps = googlemaps.Client(key=os.getenv("GOOGLE_MAPS_KEY"))

[2026-01-12 18:19:02,121: INFO: client: API queries_quota: 60]


In [6]:
class DataProcessor:
    def __init__(self, config: DataProcessingConfig):
        self.config = config
        self.train = pd.read_csv(self.config.train)
        self.test = pd.read_csv(self.config.test)

    def code(self, x):
        """Geocode a location x using Google Maps API"""
        try:
            result = maps.geocode(x.lower(), region="gh")
            if result:
                lat = result[0]["geometry"]["location"]["lat"]
                lng = result[0]["geometry"]["location"]["lng"]
                return (lat, lng)
        except Exception as e:
            logger.info(f"Error geocoding {x}: {e}")
        return (None, None)

    def create_cache(self, df):
        """Save geocode data as json"""
        cache = {}
        locs = df["loc"].unique()
        for i, loc in enumerate(locs, 1):
            lat, lng = self.code(loc)
            cache[loc] = {"lat": lat, "lng": lng}

        save_json(self.config.geocode_cache, cache)
        return cache

    def get_lat_lng(self, location, geocodes_cache):
        """Fetch the latitude and longitude of a location"""
        location_lower = location.lower()
        if location_lower in geocodes_cache:
            result = geocodes_cache[location_lower]
            return (result["lat"], result["lng"])
        else:
            lat, lng = self.code(location_lower)
            geocodes_cache[location_lower] = {"lat": lat, "lng": lng}
            save_json(self.config.geocode_cache, geocodes_cache)
            return (lat, lng)

    def drop_duplicates(self):
        self.train = self.train.drop_duplicates(subset="url")
        self.test = self.test.drop_duplicates(subset="url")

        return self

    def trim_price(self):
        x = np.log(self.train["price"])
        q1, q3 = np.percentile(x, [25, 75])
        iqr = q3 - q1
        l1 = q1 - 1.5 * iqr
        l2 = q3 + 1.5 * iqr

        train_outliers = self.train[
            (np.log(self.train["price"]) < l1) | (np.log(self.train["price"]) > l2)
        ]

        test_outliers = self.test[
            (np.log(self.test["price"]) < l1) | (np.log(self.test["price"]) > l2)
        ]

        logger.info(
            f"{len(train_outliers)} outliers in training set and {len(test_outliers)} outliers in evaluation set."
        )

        self.train = self.train.drop(index=train_outliers.index)
        self.test = self.test.drop(index=test_outliers.index)

        return self

    def clean_condition(self):
        self.train.Condition = self.train.Condition.map(
            {
                "Newly-Built": "New",
                "Fairly Used": "Used",
                "Old": "Used",
                "Renovated": "Renovated",
            }
        )
        self.test.Condition = self.test.Condition.map(
            {
                "Newly-Built": "New",
                "Fairly Used": "Used",
                "Old": "Used",
                "Renovated": "Renovated",
            }
        )
        return self

    def select_columns(self):
        drop_list = ["fetch_date", "Property Size", "locality_grouped"]
        self.train = self.train.drop(columns=drop_list)
        self.test = self.test.drop(columns=drop_list)
        return self

    def rename_columns(self, data):
        for col in data.columns.to_list():
            data.rename(
                columns={col: col.lower().replace(" ", "_").replace("-", "_")},
                inplace=True,
            )
        return data

    def save(self):
        self.train.to_csv(
            os.path.join(self.config.root_dir, "preprocessed_train.csv"), index=False
        )
        self.test.to_csv(
            os.path.join(self.config.root_dir, "preprocessed_eval.csv"), index=False
        )

    def process(self):
        try:
            geocodes_cache = load_json(self.config.geocode_cache)
        except FileNotFoundError:
            self.config.geocode_cache.parent.mkdir(parents=True, exist_ok=True)
            geocodes_cache = self.create_cache(self.config.data_dir)

        self.train[["lat", "lng"]] = self.train["loc"].apply(
            lambda x: pd.Series(self.get_lat_lng(x, geocodes_cache))
        )
        self.test[["lat", "lng"]] = self.test["loc"].apply(
            lambda x: pd.Series(self.get_lat_lng(x, geocodes_cache))
        )

        self.drop_duplicates()
        self.trim_price()
        self.clean_condition()
        self.select_columns()
        self.train = self.rename_columns(self.train)
        self.test = self.rename_columns(self.test)
        self.save()

In [7]:
try:
    config = ConfigurationManager()
    data_processsing_config = config.get_data_processing_config()
    data_processing = DataProcessor(config=data_processsing_config)
    data_processing.process()
except Exception as e:
    raise e

[2026-01-12 18:19:02,190: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-01-12 18:19:02,195: INFO: common: yaml file: params.yaml loaded successfully]
[2026-01-12 18:19:02,201: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-01-12 18:19:02,204: INFO: common: Created directory at: artifacts]
[2026-01-12 18:19:02,208: INFO: common: Created directory at: artifacts/pipeline/data_preprocess]
[2026-01-12 18:19:02,371: INFO: common: JSON file loaded successfully from: artifacts/cache/geocode_cache.json]
[2026-01-12 18:19:04,692: INFO: 3395242532: 45 outliers in training set and 16 outliers in evaluation set.]
